# Трансформеры и векторные представления текста

В предыдущей работе мы использовали рекуррентные сети для классификации текстов. Однако классификация — далеко не единственная задача обработки естественного языка. Во многих случаях требуется определить, насколько похожи два текста по смыслу, найти близкие документы или сгруппировать их по содержанию. Для решения таких задач необходимо научиться представлять текст в виде числового вектора, сохраняющего его смысл.

### Представление текста: первые подходы
Возьмём пять коротких предложений:

In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

sentences = [
    "The programmer implemented a binary tree in the system.",
    "The developer created a hierarchical data structure for the application.",
    "The gardener planted a new tree near the irrigation system.",
    "The programmer fixed a software bug.",
    "The engineer built a monitoring system for the network."
]

Интуитивно понятно, что первые два предложения близки по смыслу, а третье описывает совершенно другую ситуацию. Четвёртое снова касается работы програмиста, но здесь он решает другую задачу. Пятое занимает промежуточное положение, но тематически оно ближе к технологиям, чем к садоводству. Давайте проверим, смогут ли знакомые нам методы векторного представления текста уловить эти нюансы.

### Мешок слов
Получим такие представления с помощью CountVectorizer и сравним предложения по косинусному сходству.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(sentences)

similarity_bow = pd.DataFrame(
    cosine_similarity(X_bow),
    index=[f"S{i}" for i in range(1, len(sentences) + 1)],
    columns=[f"S{i}" for i in range(1, len(sentences) + 1)],
)

similarity_bow.round(3).style.background_gradient(cmap="Blues")

,S1,S2,S3,S4,S5
S1,1.000000,0.381000,0.572000,0.424000,0.500000
S2,0.381000,1.000000,0.364000,0.270000,0.477000
S3,0.572000,0.364000,1.000000,0.270000,0.477000
S4,0.424000,0.270000,0.270000,1.000000,0.283000
S5,0.500000,0.477000,0.477000,0.283000,1.000000


Наибольшее сходство получили предложения, имеющие много общих слов. Например, первое и третье предложения оказались наиболее похожими из-за слов **tree** и **system**, хотя говорят о совершенно разных объектах. Напротив, пары из первого и второго, а также первого и четвёртого предложений, описывающие близкие по смыслу ситуации, получили невысокую оценку сходства, посколько содержат мало совпадающих слов.

Попробуем уменьшить влияние слишком часто встречающихся слов.

### TF-IDF

В модели TF-IDF каждому слову назначается вес, зависящий не только от его частоты в документе, но и от того, насколько редко оно встречается в коллекции текстов. Благодаря этому общеупотребительные слова оказывают меньшее влияние на оценку сходства.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(sentences)

similarity_tfidf = pd.DataFrame(
    cosine_similarity(X_tfidf),
    index=similarity_bow.index,
    columns=similarity_bow.columns,
)

similarity_tfidf.round(3).style.background_gradient(cmap="Blues")

,S1,S2,S3,S4,S5
S1,1.000000,0.139000,0.319000,0.236000,0.233000
S2,0.139000,1.000000,0.125000,0.084000,0.231000
S3,0.319000,0.125000,1.000000,0.087000,0.209000
S4,0.236000,0.084000,0.087000,1.000000,0.094000
S5,0.233000,0.231000,0.209000,0.094000,1.000000


In [4]:
from itertools import combinations

pairs = [
    {
        "Pair": f"S{i+1} ↔ S{j+1}",
        "BoW": similarity_bow.iloc[i, j],
        "TF-IDF": similarity_tfidf.iloc[i, j],
        "Text 1": f"S{i+1}: {sentences[i]}",
        "Text 2": f"S{j+1}: {sentences[j]}",
    }
    for i, j in combinations(range(len(sentences)), 2)
]

comparison = pd.DataFrame(pairs)

comparison["BoW rank"] = (
    comparison["BoW"].rank(method="first", ascending=False).astype(int)
)

comparison["TF-IDF rank"] = (
    comparison["TF-IDF"].rank(method="first", ascending=False).astype(int)
)

comparison["Rank Δ"] = comparison["BoW rank"] - comparison["TF-IDF rank"]

comparison = (
    comparison[["Pair", "BoW rank", "TF-IDF rank", "Rank Δ",
                "BoW", "TF-IDF", "Text 1", "Text 2"]].sort_values("BoW rank")
                .round(3)
)

(
    comparison.style
    .set_properties(subset=["Text 1", "Text 2"], **{"text-align": "left"})
    .format({"BoW": "{:.3f}", "TF-IDF": "{:.3f}"})
)

,Pair,BoW rank,TF-IDF rank,Rank Δ,BoW,TF-IDF,Text 1,Text 2
1,S1 ↔ S3,1,1,0,0.572,0.319,S1: The programmer implemented a binary tree in the system.,S3: The gardener planted a new tree near the irrigation system.
3,S1 ↔ S5,2,3,-1,0.500,0.233,S1: The programmer implemented a binary tree in the system.,S5: The engineer built a monitoring system for the network.
6,S2 ↔ S5,3,4,-1,0.477,0.231,S2: The developer created a hierarchical data structure for the application.,S5: The engineer built a monitoring system for the network.
8,S3 ↔ S5,4,5,-1,0.477,0.209,S3: The gardener planted a new tree near the irrigation system.,S5: The engineer built a monitoring system for the network.
2,S1 ↔ S4,5,2,3,0.424,0.236,S1: The programmer implemented a binary tree in the system.,S4: The programmer fixed a software bug.
0,S1 ↔ S2,6,6,0,0.381,0.139,S1: The programmer implemented a binary tree in the system.,S2: The developer created a hierarchical data structure for the application.
4,S2 ↔ S3,7,7,0,0.364,0.125,S2: The developer created a hierarchical data structure for the application.,S3: The gardener planted a new tree near the irrigation system.
9,S4 ↔ S5,8,8,0,0.283,0.094,S4: The programmer fixed a software bug.,S5: The engineer built a monitoring system for the network.
5,S2 ↔ S4,9,10,-1,0.270,0.084,S2: The developer created a hierarchical data structure for the application.,S4: The programmer fixed a software bug.
7,S3 ↔ S4,10,9,1,0.270,0.087,S3: The gardener planted a new tree near the irrigation system.,S4: The programmer fixed a software bug.


Сравним результаты между собой и с тем, как похожесть этих предложений оценил бы человек.

| Пара предложений | Интуитивная оценка | BoW | TF-IDF |
|------------------|--------------------|:---:|:------:|
| **S1 ↔ S3**<br>`binary tree` ↔ `tree` | Кажутся непохожими: совпадает слово **tree**, но смысл разный. | 🥇 1 | 🥇 1 |
| **S1 ↔ S5**<br>два программиста | Действительно похожи по смыслу. | 5 | 🥈 2 |
| **S1 ↔ S4**<br>`system` | Есть некоторое сходство, но оно строится главным образом на слове **system**. | 🥈 2 | 3 |
| **S1 ↔ S2**<br>`programmer` ↔ `developer` | Очень похожи по смыслу, но используют разные слова. | 6 | 6 |

TF-IDF действительно делает результат более разумным: уменьшается влияние часто встречающихся слов, а более информативные слова начинают играть большую роль. Следует отметить, что на маленьком наборе предложений эффект TF-IDF далеко не всегда оказывается заметным. В данном примере различия проявились достаточно хорошо, однако принципиальные ограничения обеих моделей остаются прежними: они сравнивают тексты по совпадающим словам и не учитывают их смысл.

* Предложения **1** и **2** по-прежнему считаются слабо похожими, хотя описывают практически одну и ту же ситуацию разными словами.

* Предложения **1** и **3** всё ещё оказываются наиболее похожими, хотя в одном случае речь идёт о структуре данных, а в другом — о настоящем дереве.

### Можно ли представить смысл слова?
Если статистики слов недостаточно (слова **programmer** и **developer** никак не связаны между собой), можно попробовать обучить модель представлять каждое слово небольшим числовым вектором. При этом слова, встречающиеся в похожих контекстах, будут получать похожие представления (близкие векторы).

Одной из первых популярных моделей такого типа стала **Word2Vec**. Позднее появились и другие алгоритмы обучения эмбеддингов. В библиотеке `gensim` доступны заранее обученные модели разных типов. Для демонстрации воспользуемся компактной моделью **GloVe**.

In [5]:
!pip install gensim

In [6]:
from gensim.downloader import load

model = load("glove-wiki-gigaword-50")

Библиотека `gensim` одержит готовые модели векторных представлений слов, обученные на больших текстовых корпусах.

In [7]:
for word in ["programmer", "developer", "tree", "bug"]:
    print(f"\n{word}")
    for similar, score in model.most_similar(word, topn=5):
        print(f"  {similar:15} {score:.3f}")


programmer
  programmers     0.702
  software        0.687
  computer        0.668
  prodigy         0.653
  animator        0.648

developer
  estate          0.711
  entrepreneur    0.707
  owner           0.701
  builder         0.697
  developers      0.692

tree
  trees           0.888
  pine            0.790
  flower          0.754
  oak             0.745
  green           0.745

bug
  bugs            0.852
  worm            0.773
  bites           0.744
  cat             0.737
  monster         0.734


Среди ближайших соседей оказываются слова, употребляющиеся в похожих контекстах.

In [8]:
pairs = [
    ("programmer", "developer"),
    ("programmer", "gardener"),
    ("tree", "forest"),
    ("tree", "structure"),
    ("implement", "plant"),
]

pd.DataFrame(
    {
        "Word 1": [a for a, _ in pairs],
        "Word 2": [b for _, b in pairs],
        "Similarity": [model.similarity(a, b) for a, b in pairs],
    }
).round(3)

,Word 1,Word 2,Similarity
0,programmer,developer,0.579
1,programmer,gardener,0.373
2,tree,forest,0.678
3,tree,structure,0.502
4,implement,plant,0.278


Однако пока это работает только для отдельных слов.

### Представление последовательности

Один из первых успешных подходов к решению задачи получения векторых представлений предложений и текстов основан на рекуррентных нейронных сетях.

В предыдущей работе мы обучали модель классифицировать новостные сообщения. При этом классификатор принимал решение не на основе самого текста, а ориентируясь на его внутреннее представление — скрытое состояние рекуррентной сети.

                Text
                  │
                  ▼
           Word embeddings
                  │
                  ▼
             LSTM / GRU
                  │
                  ▼
         Hidden state (vector)
                  │
                  ▼
           Linear classifier
                  │
                  ▼
             News class

Если мысленно убрать последний линейный слой, то скрытое состояние сети можно рассматривать как векторное представление всего текста. Таким образом, ещё в предыдущей работе мы фактически получали эмбеддинги предложений, хотя тогда использовали их только для классификации.


Однако такие представления строились только в процессе обучения конкретной задачи. Мы обучали сеть различать новости по категориям, а не создавать универсальные эмбеддинги текстов. Тем не менее сама идея оказалась очень перспективной: сначала получить компактное векторное представление текста, а затем использовать его для решения различных задач.

### Ограничения рекуррентных сетей

Рекуррентные сети позволили учитывать порядок слов и строить представление всей последовательности. Однако при работе с длинными текстами проявились ограничения самой архитектуры:
* слова обрабатываются последовательно;
* вычисления плохо распараллеливаются;
* чем дальше слова находятся друг от друга, тем сложнее модели установить связь между ними.

К какому слову относится местоимение **it** в следующем предложении?

In [9]:
(sentence := "The animal didn't cross the street because it was too tired.")

"The animal didn't cross the street because it was too tired."



`animal → didn't → cross → the → street → because → it → was → too → tired`

Рекуррентная сеть обрабатывает предложение слева направо. Чтобы при обработке слова **it** использовалась информацию о слове **animal**, она должна пройти через всю последовательность.



### Механизм внимания

Можно ли отказаться от последовательной обработки текста?

Вместо передачи информации от слова к слову модель может сразу рассматривать все слова предложения и самостоятельно определять, какие из них наиболее важны в данном контексте.

                   animal
                     │
                     ▼
      street ──────► it ◄─────── tired

При обработке слова **it** модель может учитывать сразу несколько слов предложения.

* **animal** — помогает понять, к кому относится местоимение;
* **street** — тоже доступно модели, но в данном случае менее важно;
* **tired** — дополнительно подсказывает, что речь идёт именно о животном, а не об улице.

В отличие от рекуррентной сети, информация не должна последовательно передаваться через все предыдущие слова. Модель сразу анализирует всю последовательность и самостоятельно определяет, какие связи наиболее важны.

### Трансформеры

Трансформер — это архитектура нейронной сети, основанная на механизме внимания (*attention*). Для каждого слова модель вычисляет веса внимания ко всем остальным словам последовательности и на их основе формирует новое контекстное представление. Благодаря этому зависимости между словами учитываются напрямую, без последовательной передачи информации через всю цепочку.

Такой подход хорошо масштабируется, эффективно использует параллельные вычисления и позволяет строить универсальные векторные представления слов, предложений и документов.

Трансформеры оказались настолько удачной архитектурой, что сегодня именно на их основе построено большинство современных моделей обработки естественного языка. Они используются для поиска похожих документов, классификации текстов, машинного перевода, генерации текста, вопросно-ответных систем и многих других задач.

При этом во всех этих случаях первым шагом часто становится построение качественного векторного представления текста — эмбеддинга.

### Предобученные модели

В предыдущих разделах курса мы каждый раз обучали или дообучали модель под конкретную задачу: классифицировали изображения, сегментировали объекты, строили рекомендательные системы или распознавали новости. Трансформеры позволяют сначала получить универсальное представление текста, а затем использовать его для решения самых разных задач.

Однако обучение современных трансформеров требует огромных корпусов текстов и значительных вычислительных ресурсов. Поэтому на практике их почти никогда не обучают с нуля. Как и в случае со свёрточными сетями или моделями семейства YOLO, обычно используют уже предобученный трансформер одним из двух основных способов:

* **дообучить (fine-tuning)** — адаптировать модель под конкретную задачу на собственных данных;
* **использовать как экстрактор признаков** — получать готовые векторные представления (эмбеддинги) текстов и применять их для поиска, кластеризации, рекомендаций и других задач.

### Получаем эмбеддинги с помощью трансформера

Для получения векторных представлений предложений воспользуемся библиотекой `sentence-transformers`. Она содержит предобученные трансформеры, специально предназначенные для построения эмбеддингов текстов.

В качестве примера используем модель **all-MiniLM-L6-v2**. Она достаточно компактна, быстро работает даже на центральном процессоре и при этом обеспечивает хорошее качество эмбеддингов. Благодаря этому модель можно запускать локально без использования GPU.

In [10]:
!pip install -q sentence-transformers

In [11]:
from sentence_transformers import SentenceTransformer

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
embeddings = model.encode(sentences)

embeddings.shape

(5, 384)

Каждое предложение превратилось в вектор из 384 чисел. Эти значения сами по себе ничего не говорят человеку, однако именно они используются для сравнения текстов по смыслу.

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

pd.DataFrame(
    similarity,
    index=[f"S{i}" for i in range(1, len(sentences) + 1)],
    columns=[f"S{i}" for i in range(1, len(sentences) + 1)],
).round(3).style.background_gradient(cmap="Blues")

,S1,S2,S3,S4,S5
S1,1.000000,0.461000,0.411000,0.550000,0.356000
S2,0.461000,1.000000,0.169000,0.255000,0.275000
S3,0.411000,0.169000,1.000000,0.177000,0.251000
S4,0.550000,0.255000,0.177000,1.000000,0.240000
S5,0.356000,0.275000,0.251000,0.240000,1.000000


In [15]:
from itertools import combinations

pairs = [
    {
        "Pair": f"S{i+1} ↔ S{j+1}",
        "Similarity": similarity[i, j],
        "Text 1": f"S{i+1}: {sentences[i]}",
        "Text 2": f"S{j+1}: {sentences[j]}",
    }
    for i, j in combinations(range(len(sentences)), 2)
]

pd.set_option("display.max_colwidth", None)

(
    pd.DataFrame(pairs)
    .sort_values("Similarity", ascending=False)
    .reset_index(drop=True)
    .style
    .set_properties(subset=["Text 1", "Text 2"], **{"text-align": "left"})
    .format({"Similarity": "{:.3f}"})
)

,Pair,Similarity,Text 1,Text 2
0,S1 ↔ S4,0.550,S1: The programmer implemented a binary tree in the system.,S4: The programmer fixed a software bug.
1,S1 ↔ S2,0.461,S1: The programmer implemented a binary tree in the system.,S2: The developer created a hierarchical data structure for the application.
2,S1 ↔ S3,0.411,S1: The programmer implemented a binary tree in the system.,S3: The gardener planted a new tree near the irrigation system.
3,S1 ↔ S5,0.356,S1: The programmer implemented a binary tree in the system.,S5: The engineer built a monitoring system for the network.
4,S2 ↔ S5,0.275,S2: The developer created a hierarchical data structure for the application.,S5: The engineer built a monitoring system for the network.
5,S2 ↔ S4,0.255,S2: The developer created a hierarchical data structure for the application.,S4: The programmer fixed a software bug.
6,S3 ↔ S5,0.251,S3: The gardener planted a new tree near the irrigation system.,S5: The engineer built a monitoring system for the network.
7,S4 ↔ S5,0.240,S4: The programmer fixed a software bug.,S5: The engineer built a monitoring system for the network.
8,S3 ↔ S4,0.177,S3: The gardener planted a new tree near the irrigation system.,S4: The programmer fixed a software bug.
9,S2 ↔ S3,0.169,S2: The developer created a hierarchical data structure for the application.,S3: The gardener planted a new tree near the irrigation system.


Полученные результаты заметно ближе к человеческому представлению о смысловой близости текстов.

* Наибольшее сходство модель обнаружила между первым и четвёртым предложениями. Оба относятся к разработке программного обеспечения, поэтому такой результат выглядит вполне закономерным.

* Второе предложение также оказалось близко к первому, несмотря на отсутствие общих ключевых слов. Модель распознала, что **binary tree** и **hierarchical data structure** обозначают связанные понятия, а слова **programmer** и **developer** часто используются в похожих контекстах.

* Предложение про садовника больше не считается наиболее похожим на первое, как это было при использовании CountVectorizer и TF-IDF.

В то же время результат нельзя назвать идеальным: всё-таки первые два предложения по смыслу ближе друг к другу, чем первое и четвёртое. Кроме того, предложение про настоящее дерево всё ещё получило сравнительно высокую оценку сходства с бинарным деревом — даже выше, чем предложение 5 об инженере.

Стоит оговориться, что для этого эксперимента мы взяли `all-MiniLM-L6-v2` — одну из самых легковесных и быстрых моделей в семействе SentenceTransformers. Она оптимизирована под скорость работы и скромные ресурсы, поэтому глубина её "понимания" контекста ограничена. Более крупные и современные трансформеры справились бы с этой задачей точнее.

Таким образом, даже относительно простые трансформеры строят значительно более содержательные векторные представления текста, чем классические статистические методы. Полученные эмбеддинги хорошо отражают семантику, однако остаются лишь приближением к реальному смыслу и не гарантируют идеального понимания каждого конкретного случая».